In [ ]:
import os
import numpy as np
import pandas as pd
import scipy
import astropy
from astropy import convolution as apc
from matplotlib import pyplot as plt
import seaborn as sns
from matplotlib.colors import LogNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable
import cooler
import bioframe
import cooltools
from cooltools.lib import plotting
import cooler as clr
import h5py
import coolbox
from coolbox.api import *
from coolbox.utilities import GenomeRange
from pygenometracks import readBed, readGtf
from pygenometracks.tracks import BedTrack
from coolbox.core.track.hist.hicfeature import *
import statannotations as annot
from statannotations.Annotator import Annotator
import xlrd

import glob
from pathlib import Path
import re

warnings.filterwarnings("ignore")
plt.set_loglevel('error')


### Setup

In [ ]:
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["font.size"] = 6

plt.rcParams.update({
    "text.usetex": False})

# makes illustrator recognize text as text objects
plt.rc('pdf',fonttype = 42) 
from matplotlib.ticker import EngFormatter
bp_formatter = EngFormatter('b')
def format_ticks(ax, x=True, y=True, rotate=True):
    if y:
        ax.yaxis.set_major_formatter(bp_formatter)
    if x:
        ax.xaxis.set_major_formatter(bp_formatter)
    if rotate:
        ax.tick_params(axis='y',rotation=45)
        ax.tick_params(axis='x',rotation=45)
        for tick in ax.get_yticklabels():
            tick.set_verticalalignment('center')

# sns.set_style('whitegrid')

# plot formatting (Hansen Lab style guide)
plt.rc("pdf", fonttype=42)
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial"]
plt.rcParams["font.size"] = 6
plt.rcParams["axes.spines.right"] = False
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.linewidth"] = 0.25
plt.rcParams["axes.titlesize"] = 6
plt.rcParams["axes.labelsize"] = 6
plt.rcParams["xtick.major.width"] = 0.25
plt.rcParams["xtick.labelsize"] = 6
plt.rcParams["ytick.major.width"] = 0.25
plt.rcParams["ytick.labelsize"] = 6
plt.rcParams["legend.fontsize"] = 6
plt.rcParams["legend.frameon"] = False
plt.rcParams["legend.facecolor"] = "none"
plt.rcParams["lines.linewidth"] = 1.00
plt.rcParams["text.usetex"] = False
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["ps.useafm"] = True

# colors (Hansen Lab style guide)
colors = {"blue": (0.000,0.467,0.733),
          "cyan": (0.200,0.733,0.933),
          "teal": (0.000,0.600,0.533),
          "orange": (0.933,0.467,0.200),
          "red": (0.800,0.200,0.067),
          "magenta": (0.933,0.200,0.467),
          "grey": (0.733,0.733,0.733)}

In [ ]:
# Differential map 

class HiCDiff2(HicMatBase):
    """
    Track for express the comparison between two HiC Track.

    Parameters
    ----------
    hic1 : coolbox.api.track.Cool
        First HiC Track or hic file path(.cool, .mcool, .hic).

    hic2 : coolbox.api.track.Cool
        Second HiC Track or hic file path(.cool, .mcool, .hic).

    args_hic: dict
        Key word arguments send to create Cool/DoctHic instance if the input hic1/hic2 is file.

    diff_method : str
        Difference method ('diff', 'log2fc'), default 'diff'


    """
    #TODO: figure out where to implement a call to process_matrix, that should do O/E normalization and hopefully fix the nonzero centering issues
    DEFAULT_ARGS_HIC = {
        "transform": "no",
        "normalize": "expect",
        "gaussian_sigma": "no",
        "process_func": "no",
    }
    
    DEFAULT_PROPERTIES = {
        'style': HicMatBase.STYLE_WINDOW,
        "cmap": "bwr",
        "transform": False,
        'diff_method': "diff",
        'args_hic': DEFAULT_ARGS_HIC,
    }

    def __init__(self, hic1, hic2, smooth=False, **kwargs):
        properties = HiCDiff.DEFAULT_PROPERTIES.copy()
       
        properties.update(kwargs)
        
        super().__init__(**properties)

        # TODO how to set hic_args in cli mode ?
        if isinstance(hic1, str):
            hic1 = HiCMat(hic1,**self.properties['args_hic'])
        if isinstance(hic2, str):
            hic2 = HiCMat(hic2)
        self.properties.update({
            'hic1': hic1,
            'hic2': hic2,
        })

        self.mat1 = None
        self.mat2 = None
        self.smooth = smooth

    def fetch_data(self, gr: GenomeRange, **kwargs) -> np.ndarray:
        # TODO find a better way
        # For cases when updating of Frame's depth_ratio leads to un equal matrix sizes for different tracks).
        args_hic = self.properties.copy()
        args_hic.update(self.properties.get('args_hic', {}))
        for hic in ('hic1', 'hic2'):
            self.properties[hic].properties.update(args_hic)

        hic1: HicMatBase = self.properties['hic1']
        hic2: HicMatBase = self.properties['hic2']
        # must set to avoid re-change GenomeRange in window style
        kwargs['gr_updated'] = True
        # use transformed gr for cases in 'window' style, other wise the gr would be transformed two times
        self.mat1 = mat1 = hic1.fetch_data(gr)
        self.mat1[self.mat1==1e-12] = np.nan
        self.mat2 = mat2 = hic2.fetch_data(gr)
        self.mat2[self.mat2==1e-12] = np.nan
        if self.smooth:
            kernel = apc.Gaussian2DKernel(2)
            self.mat1 = apc.convolve(self.mat1,kernel,boundary='extend')
            self.mat2 = apc.convolve(self.mat2,kernel,boundary='extend')
        print(f'sums before normalization: {np.nansum(self.mat1)},{np.nansum(self.mat2)}')
        self.mat1 = hic1.process_matrix(self.mat1)
        self.mat2 = hic2.process_matrix(self.mat2)
        print(f'sums after normalization: {np.nansum(self.mat1)},{np.nansum(self.mat2)}')
        # print((np.nanmedian(self.mat1),np.nanmean(self.mat1)))
        # print(np.nansum(np.concatenate((self.mat1[15,:14],self.mat1[15,17:]))),np.nansum(np.concatenate((self.mat1[50,:49],self.mat1[50,52:]))))
        # print((np.nanmedian(self.mat2),np.nanmean(self.mat2)))
        # print(np.nansum(np.concatenate((self.mat2[15,:14],self.mat2[15,17:]))),np.nansum(np.concatenate((self.mat2[50,:49],self.mat2[50,52:]))))
        diff_mat = self.diff_matrix(self.mat1, self.mat2)
        try:
            self.SMALL_VALUE = diff_mat[diff_mat > 0].min()
        except Exception:
            pass
        return diff_mat

    def diff_matrix(self, mat1, mat2):
        diff_mth = self.properties['diff_method']
        if diff_mth == 'log2fc':
            output = np.log2(mat1) - np.log2(mat2)
            print(f'Median diff: {np.nanmedian(output)}')
            print(f'Mean diff: {np.nanmean(output)}')
            mask = np.isnan(output)
            output[mask]=0
            # print(output[5,:])
            return output
            # return np.log2(mat1) - np.log2(mat2)
        elif diff_mth == 'log10fc':
            output = np.log10(mat1) - np.log10(mat2)
            print(f'Median diff: {np.nanmedian(output)}')
            print(f'Mean diff: {np.nanmean(output)}')
            mask = np.isnan(output)
            output[mask]=0
            return output
            # return np.log10(mat1)-np.log10(mat2)
        else:
            return mat1 - mat2

# define new bed format
class NewBed(Track):
    """
    Custom pygenometracks class that allows for control over the aesthetics of gene annotations in
    region plots. 
    """
    def __init__(self, coolbox_prop_dict, pygenometracks_prop_dict):
        super().__init__(coolbox_prop_dict)  # init 
        self.pygenometracks_object = BedTrack(pygenometracks_prop_dict)

    def fetch_data(self, gr, **kwargs):
        pass

    def plot(self, ax, gr, **kwargs):
#         x = gr.start + self.properties['offset'] * (gr.end - gr.start)
#         ax.text(x, 0, gr.chrom, fontsize=self.properties['fontsize'])
#         ax.set_xlim(gr.start, gr.end)
        self.pygenometracks_object.plot(ax, gr.chrom, gr.start, gr.end)
        ax.set_xlim([gr.start, gr.end])


def auto_scale_bigwigs(bigwig_list, region, y_max=None, y_min=0):
    """
    Autoscales y axis of bigwigs in bigwig list based off values in `region`.
    """
    def get_max_y_value(bigwig_list, region):
        max_y_values = []
        for bigwig in bigwig_list:
            max_y_values.append(np.amax(bigwig.fetch_plot_data(GenomeRange(region))))

        y_max = round(max(max_y_values) * 1.05)
        return y_max
    
    if y_max is None:
        if type(region) == list:
            y_max_list = [get_max_y_value(bigwig_list, single_region) for single_region in region]
            y_max = max(y_max_list)
        else:
            y_max = get_max_y_value(bigwig_list, region)
                
    return [bigwig + MaxValue(y_max) + MinValue(y_min) for bigwig in bigwig_list]

def make_bigwig_list(bigwigs, region, condition_order, bw_bins=1600, track_height=2, autoscale=True, 
                     y_max=None):
    
    """
    Helper function for `make_region_plot` that converts file paths to bigwig plotting objects.
    """
    assert len(bigwigs) % len(condition_order) == 0
    bigwig_list = [BigWig(bigwig, number_of_bins=bw_bins) + Title(title) \
                            + TrackHeight(2) for bigwig, title in zip(bigwigs, condition_order)]
    if autoscale:
        bigwig_list = auto_scale_bigwigs(bigwig_list, region, y_max=y_max)
    return bigwig_list

class ChromName(Track):
    """
    Track for show chromosome name.

    Parameters
    ----------
    fontsize : float
        Font name to show.

    offset : float
        Offset ratio to the start position.
    """

    DEFAULT_PROPERTIES = {
        "fontsize": 50,
        "offset": 0.45,
    }

    def __init__(self, **kwargs):
        properties = ChromName.DEFAULT_PROPERTIES.copy()
        properties.update(kwargs)
        super().__init__(properties)

    def fetch_data(self, gr: GenomeRange, **kwargs):
        return gr.chrom  # return chromosome name

    def plot(self, ax, gr: GenomeRange, **kwargs):
        x = gr.start + self.properties['offset'] * (gr.end - gr.start)
        ax.text(x, 0, gr.chrom, fontsize=self.properties['fontsize'])
        ax.set_xlim(gr.start, gr.end)

from coolbox.core.track.hicmat import HiCMat
class DiffIns(HicFeature):
    """
    Differential insulation score track.

    Parameters
    ----------
    hicmat: {str, HicMatBase}
        The input hicmat file or HicMatBase object used to calculate di score.

    hicmat2: {str, HicMatBase}
        Second input hicmat file or HicMatBase object; insulation of this will be subtracted from insulation of first

    window_size: {int, str}, optional
        Width of the diamond region along the matrix used to calculate di. default: 20
        window_size can also be a str with format like "20-40" representing a range of window_sizes.

    normalize: bool, optional
        Weather to log-nomalize the insulation score array. default: true

    method: {'standard', 'adaptive'}, optional
        Method used for calculating di. default: 'adaptive'

    args_hic : dict, optional
        Key words arguments used to create Cool/DotHic tack when the input hicmat is a file path
    """

    DEFAULT_PROPERTIES = {
        "style": "line",
        "cmap": "coolwarm_r",
    }

    def __init__(self,
                 hicmat: Union[str, HicMatBase],
                 hicmat2: Union[str, HicMatBase],
                 window_size: Union[int, str] = 20,
                 normalize: bool = True,
                 args_hic: dict = None,
                 **kwargs):
        properties = InsuScore.DEFAULT_PROPERTIES.copy()
        properties.update(kwargs)
        super().__init__(hicmat, args_hic, **properties)
        self.window_size = window_size
        self.normalize = normalize
        if type(hicmat2)==str:
            self.hicmat2 = HiCMat(hicmat2,**(args_hic or {}))
        elif type(hicmat2)==HicMatBase:
            self.hicmat2=hicmat2
        elif hicmat2:
            raise ValueError("hicmat2 must be string or HicMatBase")

    def fetch_data(self, gr: GenomeRange, **kwargs) -> np.ndarray:
        window_size = self.window_size
        try:
            if isinstance(window_size, int):
                window_size = [window_size]
            elif isinstance(window_size, str):
                st, ed = window_size.split("-")
                window_size = list(range(int(st), int(ed)))
            else:
                raise ValueError
        except Exception:
            raise ValueError("window_size should be like int: 12 or str: '10-50'.")

        insus = []
        # for multiple window sizes
        for ws in window_size:
            mat: np.ndarray = self.hicmat.fetch_data(gr)
            mlen = mat.shape[0]
            if mlen < ws * 2:
                insus.append(np.zeros(mlen))
            else:
                insu = np.full(mlen, np.nan, dtype=mat.dtype)
                for row in range(ws, mlen - ws):
                    sub_mat = mat[row - ws: row, row + 1: row + ws + 1]
                    insu[row] = np.nanmean(sub_mat)

                if self.normalize:
                    insu = np.log2(insu / np.nanmean(insu))
                    insu[~np.isfinite(insu)] = 0
                insus.append(insu)
        insus2 = []
        # for multiple window sizes
        for ws in window_size:
            mat: np.ndarray = self.hicmat2.fetch_data(gr)
            mlen = mat.shape[0]
            if mlen < ws * 2:
                insus2.append(np.zeros(mlen))
            else:
                insu2 = np.full(mlen, np.nan, dtype=mat.dtype)
                for row in range(ws, mlen - ws):
                    sub_mat = mat[row - ws: row, row + 1: row + ws + 1]
                    insu2[row] = np.nanmean(sub_mat)

                if self.normalize:
                    insu2 = np.log2(insu2 / np.nanmean(insu2))
                    insu2[~np.isfinite(insu2)] = 0
                insus2.append(insu2)
        diff_insus = []
        for (i1, i2) in zip(insus,insus2):
            di = np.array([score1-score2 for (score1,score2) in zip(i1,i2)])
            diff_insus.append(di)

        return diff_insus[0] if len(diff_insus) == 1 else np.array(diff_insus)[::-1]

In [ ]:
def cat_prefix_to_list(prefix, file_list):
    """
    Helper function that concatenates a prefix to every element
    in a list. 
    """
    return [os.path.join(prefix, file) for file in file_list]

# this was the original version that miles gave me
def make_region_plot(region, resolution, microc_file, microc_title,
                     chip_list_list, condition_order, highlight_region_list=[],
                     forward_proseq_list=[], reverse_proseq_list=[], 
                     bigwig_bins=3600, highlight_buffer=1000, pro_bins=None):
    
    """
    Function that makes a plot of a region with Micro-C, ChIP-seq, and Pro-seq.
    
    region: (str) string that specifies region for plotting.
    resolution: (int) resolution of micro-c map used
    highlight_region_list: List of regions of format [chr, start, end] to highlight
    microc_file: (str) file path to micro-c data
    microc_title: (str) title for micro-c
    chip_list_list: List of lists containing paths to bigwigs
    forward_proseq_list: List of paths to forward strand pro-seq files
    reverse_proseq_list: List of paths to reverse strand pro-seq files
    condition_order: List indicating order of conditions to plot. 
    """
    
    heatmap = Cool(microc_file, resolution=resolution, **MICROC_PLOTTING_PARAMS) + Title(microc_title)
    frame = heatmap
    if pro_bins is None:
        pro_bins = bigwig_bins
    def iterate_over_datasets(frame):
        i = 0 
        for chip_list in chip_list_list:
            bigwig_list = make_bigwig_list(chip_list, region, condition_order, 
                                           DEFAULT_COLOR_LIST[i%len(DEFAULT_COLOR_LIST)], bw_bins=bigwig_bins)
            i += 1
            for bigwig in bigwig_list:
                frame += bigwig + Spacer(SPACER_HEIGHT)
        f_bigwig_list = make_bigwig_list(forward_proseq_list, region, condition_order, 
                                       DEFAULT_COLOR_LIST[i%len(DEFAULT_COLOR_LIST)], bw_bins=pro_bins,  
                                         track_height=PRO_SEQ_TRACK_HEIGHT)
        r_bigwig_list = make_bigwig_list(reverse_proseq_list, region, len(reverse_proseq_list)*[""], 
                               DEFAULT_COLOR_LIST[(i+1)%len(DEFAULT_COLOR_LIST)], bw_bins=pro_bins, 
                                         track_height=PRO_SEQ_TRACK_HEIGHT)
        r_bigwig_list = [r_bigwig + Inverted() for r_bigwig in r_bigwig_list]
        for f_bigwig, r_bigwig in zip(f_bigwig_list, r_bigwig_list):
                frame += f_bigwig + r_bigwig + Spacer(SPACER_HEIGHT)
        annotations = NewBed(*ANNOTATIONS_PARAMS)
        frame += annotations
        return frame
    
    # check if it needs a highlight
    if len(highlight_region_list) > 0:
        highlight_regions = []
        for highlight_region in highlight_region_list:
            highlight_region[1] -= highlight_buffer
            highlight_region[2] += highlight_buffer 
            highlight_regions.append(tuple(highlight_region))

        highlights = HighLights(highlight_regions, **HIGHLIGHT_PARAMS)
        with highlights:
            frame = iterate_over_datasets(frame)
    else:
        frame = iterate_over_datasets(frame)
    #frame += XAxis()
    return frame

# this is the version with my modifications
def make_region_plot_mod(region, resolution, microc_file,
                     bw_list, condition_order,
                     file_bottom=None, cmap_chip='tab10', bigwig_bins=1600, autoscale=True, plot_bed=False,
                        variant_track_height=2, custom_valranges=None):
    
    """
    Function that makes a plot of a region with Micro-C, ChIP-seq, and Pro-seq.
    
    region: (str) string that specifies region for plotting.
    resolution: (int) resolution of micro-c map used
    highlight_region_list: List of regions of format [chr, start, end] to highlight
    microc_file: (str) file path to micro-c data
    microc_title: (str) title for micro-c
    chip_list_list: List of lists containing paths to bigwigs
    forward_proseq_list: List of paths to forward strand pro-seq files
    reverse_proseq_list: List of paths to reverse strand pro-seq files
    condition_order: List indicating order of conditions to plot. 
    """
    print(region, resolution, microc_file)
    heatmap = Cool(microc_file, file_bottom=file_bottom, resolution=resolution, **MICROC_PLOTTING_PARAMS) 
    
    frame = heatmap + Spacer(SPACER_HEIGHT*3)
    
    # if the bed has a strand column, i believe it should plot as triangles
    if plot_bed:
        frame+=NewBed(*BED_PARAMS) + TrackHeight(4) + Spacer(SPACER_HEIGHT)
   
    if len(bw_list)>0:
        bigwig_list = make_bigwig_list(bw_list, region, condition_order, autoscale=False, bw_bins=bigwig_bins)
        cmap_arr = sns.color_palette(cmap_chip, n_colors=len(bigwig_list))

        for i, bigwig in enumerate(bigwig_list):
            if autoscale:
                frame += auto_scale_bigwig(bigwig, region) + Color(cmap_arr[i]) + Spacer(SPACER_HEIGHT)
            else:
                frame += bigwig + MaxValue(custom_valranges[i]) + MinValue(0) + Color(cmap_arr[i]) + Spacer(SPACER_HEIGHT)

    # gene annotations
    annotations = NewBed(*ANNOTATIONS_PARAMS)
    frame+=annotations

    return frame

def auto_scale_bigwig(bigwig_single, region, y_max=None, y_min=0):
    """
    Autoscales y axis of bigwigs in bigwig list based off values in `region`.
    """
    def get_max_y_value(bigwig_single, region):
        return np.round(np.amax(bigwig_single.fetch_plot_data(GenomeRange(region))))
    
    if y_max is None:
        if type(region) == list:
            y_max_list = [get_max_y_value(bigwig_single, single_region) for single_region in region]
            y_max = max(y_max_list)
        else:
            y_max = get_max_y_value(bigwig_single, region)
    
    return bigwig_single + MaxValue(y_max) + MinValue(y_min)
            
def auto_scale_bigwigs(bigwig_list, region, y_max=None, y_min=0):
    """
    Autoscales y axis of bigwigs in bigwig list based off values in `region`.
    """
    def get_max_y_value(bigwig_list, region):
        max_y_values = []
        for bigwig in bigwig_list:
            max_y_values.append(np.amax(bigwig.fetch_plot_data(GenomeRange(region))))

        y_max = round(max(max_y_values) * 1.05)
        return y_max
    
    if y_max is None:
        if type(region) == list:
            y_max_list = [get_max_y_value(bigwig_list, single_region) for single_region in region]
            y_max = max(y_max_list)
        else:
            y_max = get_max_y_value(bigwig_list, region)
                
    return [bigwig + MaxValue(y_max) + MinValue(y_min) for bigwig in bigwig_list]

def read_cooler(cooler_path, resolution=250):
    """
    Simple function wrapper to read cooler file at a particular resolution.
    """
    if clr.fileops.is_cooler(cooler_path):
        return clr.Cooler(cooler_path)
    elif clr.fileops.is_multires_file(cooler_path):
        return clr.Cooler(cooler_path + "::resolutions/" + str(int(resolution)))

# define new bed format
class NewBed(Track):
    """
    Custom pygenometracks class that allows for control over the aesthetics of gene annotations in
    region plots. 
    """
    def __init__(self, coolbox_prop_dict, pygenometracks_prop_dict):
        super().__init__(coolbox_prop_dict)  # init 
        self.pygenometracks_object = BedTrack(pygenometracks_prop_dict)

    def fetch_data(self, gr, **kwargs):
        pass

    def plot(self, ax, gr, **kwargs):
#         x = gr.start + self.properties['offset'] * (gr.end - gr.start)
#         ax.text(x, 0, gr.chrom, fontsize=self.properties['fontsize'])
#         ax.set_xlim(gr.start, gr.end)
        self.pygenometracks_object.plot(ax, gr.chrom, gr.start, gr.end)
        ax.set_xlim([gr.start, gr.end])
        
def make_bigwig_list(bigwigs, region, condition_order, bw_bins=1600, track_height=2, autoscale=True, 
                     y_max=None):
    
    """
    Helper function for `make_region_plot` that converts file paths to bigwig plotting objects.
    """
    assert len(bigwigs) % len(condition_order) == 0
    bigwig_list = [BigWig(bigwig, number_of_bins=bw_bins) + Title(title) \
                            + TrackHeight(2) for bigwig, title in zip(bigwigs, condition_order)]
    if autoscale:
        bigwig_list = auto_scale_bigwigs(bigwig_list, region, y_max=y_max)
    return bigwig_list

In [ ]:
# Define plotting aesthetics as global variables
HIGHLIGHT_PARAMS = {"color":"gray",
                   "alpha":0.5,
                   "border_line":False}

MICROC_PLOTTING_PARAMS = {"style":"matrix",
                          "transform":"log10",
                          "depth_ratio":"full",
                          "balance":'weight',
                          "cmap":"fall",
                          "max_value":-1.5,
                          "min_value":-3.5}
INS_PARAMS = {'color': "#ee3377",'line_width':0.5}

SPACER_HEIGHT = 0.4
DEFAULT_COLOR_LIST = ["#FF0000", "#00FA00", "#0000FF", "#A0F000"]
PRO_SEQ_TRACK_HEIGHT = 1.2

# change this to the location of your genome annotations
GTF_FILENAME = "/mnt/md0/Domenic/genes/mm39/mm39_NCBI_RefSeq_names_unique.bed" 
ANNOTATIONS_PARAMS = [{"height":15},
                      {"file": GTF_FILENAME, # PATH TO BED FILE OF GENE ANNOTATIONS
                       "max_labels":1000,
                       "merge_transcripts":True, 
                       "gene_rows":30}]

# change this to your region of course
regions = {
        'mm39':'chr18:58032072-59034072',
	'mm10':'chr18:57899000-58901000',
	'mm9':'chr18:58005490-59007490'
}
# regions={'mm39':'chr18:58032072-59034072'}
incl= ['ATAC', 'H3K27ac','RNASeq', 'RNASeq2']

### Plot RCMC

In [ ]:
paths = {'cool1':'path/to/mcool1.mcool',
            'cool2':'path/to/mcool2.mcool'}
def get_cools(paths,resolution):
	cools = {}
	for name,path in paths.items():
		c = cooler.Cooler(path+'::resolutions/'+str(resolution))
		cools[name]=c
	return cools
cools = get_cools(paths,5000)

In [ ]:
# plot a raw contact map, with one condition above the diagonal and one below
MICROC_PLOTTING_PARAMS = {"style":"matrix",
                          "transform":"log10",
                          "depth_ratio":"full",
                          "balance":'weight',
                          "cmap":"fall",
                          "max_value":-1.75,
                          "min_value":-3.75}
frame = Cool(paths['cool2'],file_bottom=paths['cool1'],resolution=1000,**MICROC_PLOTTING_PARAMS)+Spacer(SPACER_HEIGHT*3)
fig = frame.plot(regions['mm39'],regions['mm39'])
fig.savefig('./raw_contact_map.svg',transparent=True,bbox_inches='tight')
fig

In [ ]:
# plot a differential contact map, with insulation track
HICDIFF_ARGS={"transform": "no",
        "normalize": "expect",
        "gaussian_sigma": "no",
        "process_func": "no",
		'balance':'weight'}
frame = HiCDiff2(paths['cool2'],
				 paths['cool1'],
				 smooth=True,resolution=1000,
				 style='matrix',diff_method='log10fc',cmap='bwr',max_value=0.75,min_value=-0.75,transform='linear',args_hic=HICDIFF_ARGS)+Spacer(SPACER_HEIGHT*3)
frame+=DiffIns(hicmat=paths['cool1'],
			   hicmat2=paths['cool2'],window_size=50,
			   args_hic={'resolution':1000,'balance':'weight'},color='#ee3377')+HLines(values=0)+Title(u"-Δins")+Spacer(SPACER_HEIGHT)
frame+=NewBed(*ANNOTATIONS_PARAMS)
fig = frame.plot(regions['mm39'])
fig.savefig('./diff_contact_map.svg',
			transparent=True,bbox_inches='tight')
fig

In [ ]:
# plot bigwigs (ChIP-seq, ATAC-seq)
bigwigs = ['path/to/bigwig1.bw',
                 'path/to/bigwig2.bw',
                 'path/to/bigwig3.bw']
cond_order = ['cond1','cond2','cond3']
region = 'chr18:58032072-59034072'
bw_list = make_bigwig_list(bigwigs,region,cond_order,autoscale=False)
for i, bw in enumerate(bw_list):
    if i==0:
        frame = auto_scale_bigwig(bw,region,y_max=36)+Color(sns.color_palette('tab10')[i])+Spacer(SPACER_HEIGHT)
    else:
        frame += auto_scale_bigwig(bw,region,y_max=36)+Color(sns.color_palette('tab10')[i])+Spacer(SPACER_HEIGHT)
frame+=XAxis(height=2.5)
frame+=NewBed(*ANNOTATIONS_PARAMS)

fig = frame.plot(region)
fig.savefig('./ChIP.svg',
            bbox_inches='tight',transparent=True)
fig

### Loop strength analysis

In [ ]:
anchors = pd.read_csv('./loop_calls.csv')
strengths = pd.read_csv('./Fbn2_loop_strengths_AbLE_byrep_20260422.csv')
anchors.rename({'end1':'start2'},axis='columns',inplace=True)
anchors['start1']-=2500
anchors['end1']=anchors['start1']+5000
anchors = anchors.iloc[:,[0,1,3,2]]
anchors['start2']-=2500
anchors['end2']=anchors['start2']+5000
anchors['name'] = strengths['loopname']
anchors.to_csv('Fbn2_loop_calls.bedpe',sep='\t',header=False,index=False)

In [ ]:
strengths.drop(columns=['Unnamed: 0'],inplace=True)
strengths['Loop ID'] = np.arange(9)+1
plot_df = strengths.melt(id_vars=['Loop ID','loopname','anchors'],var_name='Condition',value_name='AbLE Score')
plot_df['Condition'].loc[(plot_df['Condition']=='u1')|(plot_df['Condition']=='u2')]='untreated'
plot_df['Condition'].loc[(plot_df['Condition']=='t1')|(plot_df['Condition']=='t2')]='+ABA'
plot_df['Condition'].loc[(plot_df['Condition']=='c1')|(plot_df['Condition']=='c2')]='\u0394CTCF'
plot_df['Condition'].loc[(plot_df['Condition']=='ct1')|(plot_df['Condition']=='ct2')]='\u0394CTCF+ABA'
plot_df['Condition'].loc[(plot_df['Condition']=='r1')|(plot_df['Condition']=='r2')]='\u0394RAD21'
plot_df['Condition'].loc[(plot_df['Condition']=='rt1')|(plot_df['Condition']=='rt2')]='\u0394RAD21+ABA'
plot_df['Condition'].loc[(plot_df['Condition']=='rc1')|(plot_df['Condition']=='rc2')]='\u0394extrusion'
plot_df['Condition'].loc[(plot_df['Condition']=='rct1')|(plot_df['Condition']=='rct2')]='\u0394extrusion+ABA'
plot_df['Condition'].loc[(plot_df['Condition']=='trp1')|(plot_df['Condition']=='trp2')]='+ABA+triptolide'

In [ ]:
# plot loop strengths across all conditions (Fig. S3b)
fig,ax = plt.subplots(1,1,figsize=(3,1.5))
sns.barplot(plot_df,
			x='Loop ID',y='AbLE Score',hue='Condition',orient='v',ax=ax,err_kws={'linewidth':0.5},alpha=0.6,legend=False,saturation=1)
sns.stripplot(plot_df,
			  x='Loop ID',y='AbLE Score',hue='Condition',orient='v',ax=ax,dodge=True,size=2,jitter=False)
plt.gca().legend().set_title(None)
plt.savefig('/mnt/md0/Jamie/Fbn2_paper/Figures/Supplemental/all_loop_strengths.svg',
			transparent=True,bbox_inches='tight')
plt.show()

In [ ]:
fig,ax = plt.subplots(1,1,figsize=(1.5,1.5))
ax.set_aspect('equal')
ax.plot(np.linspace(-.07,.07),[0]*50,'--k',alpha=0.4)
ax.plot([0]*50,np.linspace(-.07,.07),'--k',alpha=0.4)
ax.scatter(y=strengths['\u0394extrusion'].loc[strengths['anchors']=='CC'],
			x=strengths['+ABA'].loc[strengths['anchors']=='CC'],
			c='#CC3311',label='CTCF-CTCF',s=5)
ax.scatter(y=strengths['\u0394extrusion'].loc[strengths['anchors']=='PC'],
			x=strengths['+ABA'].loc[strengths['anchors']=='PC'],
			c='#009988',label='cCRE-CTCF',s=5)
ax.scatter(y=strengths['\u0394extrusion'].loc[strengths['anchors']=='PP'],
			x=strengths['+ABA'].loc[strengths['anchors']=='PP'],
			c='#0077bb',label='cCRE-cCRE',s=5)
ax.legend()
ax.set_xticks([-.06,-.04,-.02,0,.02,.04,.06])
ax.set_yticks([-.06,-.04,-.02,0,.02,.04,.06])
ax.set_xticklabels(ax.get_xticklabels(),rotation=45,verticalalignment='top')
ax.set_xlabel('\u0394AbLE score (+ABA - unt)')
ax.set_ylabel('\u0394AbLE score (\u0394extrusion - unt)')
ax.set_xlim(-0.07,0.07)
ax.set_ylim(-0.07,0.07)
for i in range(len(strengths['anchors'])):
	ax.annotate(str(i+1),(strengths['+ABA'].iloc[i],
					   strengths['\u0394extrusion'].iloc[i]),
					   horizontalalignment='left',
					   verticalalignment='bottom')
plt.savefig('./Figure_2D.svg',transparent=True,bbox_inches='tight')
plt.show()

In [ ]:
fig,ax = plt.subplots(1,1,figsize=(1.5,1.5))
ax.set_aspect('equal')
ax.plot(np.linspace(-.07,.07),[0]*50,'--k',alpha=0.4)
ax.plot([0]*50,np.linspace(-.07,.07),'--k',alpha=0.4)
ax.scatter(y=strengths['+ABA+trp'].loc[strengths['anchors']=='CC'],
			x=strengths['+ABA'].loc[strengths['anchors']=='CC'],
			c='#CC3311',label='CTCF-CTCF',s=5)
ax.scatter(y=strengths['+ABA+trp'].loc[strengths['anchors']=='PC'],
			x=strengths['+ABA'].loc[strengths['anchors']=='PC'],
			c='#009988',label='cCRE-CTCF',s=5)
ax.scatter(y=strengths['+ABA+trp'].loc[strengths['anchors']=='PP'],
			x=strengths['+ABA'].loc[strengths['anchors']=='PP'],
			c='#0077bb',label='cCRE-cCRE',s=5)
ax.legend()
ax.set_xticks([-.06,-.04,-.02,0,.02,.04,.06])
ax.set_yticks([-.06,-.04,-.02,0,.02,.04,.06])
ax.set_xticklabels(ax.get_xticklabels(),rotation=45,va='top')
ax.set_xlabel('\u0394AbLE score (+ABA - unt)')
ax.set_ylabel('\u0394AbLE score (+ABA+trp - unt)')
ax.set_xlim(-0.07,0.07)
ax.set_ylim(-0.07,0.07)
for i in range(len(strengths['anchors'])):
	ax.annotate(str(i+1),(strengths['+ABA'].iloc[i],
					   strengths['+ABA+trp'].iloc[i]),
					   horizontalalignment='left',
					   verticalalignment='bottom')
plt.savefig('./Figure_2F.svg',transparent=True,bbox_inches='tight')
plt.show()